# Task 17: Robustness, model decision, TMD validation and error analysis

本 notebook 完成四项任务：

1. 对 Task 15/16 做 paired gene-cluster bootstrap，报告 AUROC、AUPRC 及模型差值的 95% CI；
2. 按预先写明的规则给出主模型建议；
3. 检查 TMD 增益在 folds、TMD/non-TMD proteins 和 genes 中是否稳定；
4. 对 64D XGBoost OOF predictions 做 exploratory error analysis。

Bootstrap 以 `Gene` 为抽样单位，保留同一 gene 内 variants 的相关性。Task 15 的 pooled OOF 是主要稳健性证据；Task 16 的单次 test bootstrap 仅量化该 test subset 内的不确定性。

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

warnings.filterwarnings("ignore")

BASE = Path("/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial")
TASK15_OOF = BASE / "task15_full64_oof.csv"
TASK16_TEST = BASE / "task16_holdout_predictions.csv"
TMD_CSV = BASE / "tmd_features.csv"
FEATURES_CSV = BASE / "features_v3.csv"

BOOTSTRAP_REPLICATES = 2000
RANDOM_STATE = 42
BOOTSTRAP_CSV = BASE / "task17_cluster_bootstrap.csv"
SUMMARY_CSV = BASE / "task17_bootstrap_summary.csv"
TMD_ROBUSTNESS_CSV = BASE / "task17_tmd_robustness.csv"
ERROR_SUMMARY_CSV = BASE / "task17_error_summary.csv"
ERROR_ROWS_CSV = BASE / "task17_error_rows.csv"

task15 = pd.read_csv(TASK15_OOF)
task16 = pd.read_csv(TASK16_TEST)
tmd = pd.read_csv(TMD_CSV)
features = pd.read_csv(FEATURES_CSV)

required15 = ["KEY", "Gene", "Mislocalized", "fold", "final_alphamissense_score", "oof_stability_61", "oof_stability_tmd_64"]
required16 = ["KEY", "Gene", "Mislocalized", "final_alphamissense_score", "xgboost_64", "tabpfn_64", "ensemble_64"]
assert all(c in task15.columns for c in required15) and task15["KEY"].is_unique
assert all(c in task16.columns for c in required16) and task16["KEY"].is_unique
assert tmd["KEY"].is_unique and "in_TMD" in tmd.columns
assert features["KEY"].is_unique
assert len(task15) == 2179 and int(task15["Mislocalized"].sum()) == 236
print(f"Task 15: n={len(task15)}, genes={task15['Gene'].nunique()}, positives={task15['Mislocalized'].sum()}")
print(f"Task 16 test: n={len(task16)}, genes={task16['Gene'].nunique()}, positives={task16['Mislocalized'].sum()}")

Task 15: n=2179, genes=871, positives=236
Task 16 test: n=210, genes=88, positives=23


## 17.1 Gene-cluster bootstrap

每次有放回抽取 genes；被重复抽到的 gene 也重复加入其全部 variants。模型差值在同一个 bootstrap sample 内计算，因此是 paired comparison。若某次抽样只有一个 class，该 replicate 被跳过。

In [2]:
def safe_metrics(y_true, score):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    valid = np.isfinite(score)
    y_valid = y_true[valid]
    score_valid = score[valid]
    if len(y_valid) == 0 or np.unique(y_valid).size < 2:
        return np.nan, np.nan, len(y_valid)
    return roc_auc_score(y_valid, score_valid), average_precision_score(y_valid, score_valid), len(y_valid)

def cluster_bootstrap(data, prediction_cols, comparison_pairs, scope, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    grouped_indices = {gene: idx for gene, idx in data.groupby("Gene").indices.items()}
    genes = np.array(list(grouped_indices), dtype=object)
    y_all = data["Mislocalized"].to_numpy(dtype=int)
    predictions = {name: data[col].to_numpy(dtype=float) for name, col in prediction_cols.items()}
    records = []
    for replicate in range(n_boot):
        sampled_genes = rng.choice(genes, size=len(genes), replace=True)
        sampled_idx = np.concatenate([grouped_indices[g] for g in sampled_genes])
        y_boot = y_all[sampled_idx]
        metric_values = {}
        for model, values in predictions.items():
            auroc, auprc, n_valid = safe_metrics(y_boot, values[sampled_idx])
            metric_values[model] = {"auroc": auroc, "auprc": auprc}
            records.append({"scope": scope, "replicate": replicate, "kind": "model", "name": model, "n": n_valid, "auroc": auroc, "auprc": auprc})
        for left, right in comparison_pairs:
            records.append({
                "scope": scope, "replicate": replicate, "kind": "difference",
                "name": f"{left}_minus_{right}", "n": len(sampled_idx),
                "auroc": metric_values[left]["auroc"] - metric_values[right]["auroc"],
                "auprc": metric_values[left]["auprc"] - metric_values[right]["auprc"],
            })
    return pd.DataFrame(records)

boot15 = cluster_bootstrap(
    task15,
    prediction_cols={"stability_61": "oof_stability_61", "stability_tmd_64": "oof_stability_tmd_64"},
    comparison_pairs=[("stability_tmd_64", "stability_61")],
    scope="task15_full_oof", n_boot=BOOTSTRAP_REPLICATES, seed=RANDOM_STATE,
)

paired15 = task15.loc[task15["final_alphamissense_score"].notna()].copy()
boot15_am = cluster_bootstrap(
    paired15,
    prediction_cols={"alphamissense": "final_alphamissense_score", "stability_tmd_64": "oof_stability_tmd_64"},
    comparison_pairs=[("stability_tmd_64", "alphamissense")],
    scope="task15_paired_am", n_boot=BOOTSTRAP_REPLICATES, seed=RANDOM_STATE + 1,
)

paired16 = task16.loc[task16["final_alphamissense_score"].notna()].copy()
boot16 = cluster_bootstrap(
    paired16,
    prediction_cols={"alphamissense": "final_alphamissense_score", "xgboost_64": "xgboost_64", "tabpfn_64": "tabpfn_64", "ensemble_64": "ensemble_64"},
    comparison_pairs=[("xgboost_64", "alphamissense"), ("tabpfn_64", "xgboost_64"), ("ensemble_64", "xgboost_64")],
    scope="task16_paired_test", n_boot=BOOTSTRAP_REPLICATES, seed=RANDOM_STATE + 2,
)

bootstrap = pd.concat([boot15, boot15_am, boot16], ignore_index=True)
bootstrap.to_csv(BOOTSTRAP_CSV, index=False)
summary = bootstrap.groupby(["scope", "kind", "name"])[["auroc", "auprc"]].agg(["mean", lambda x: x.quantile(0.025), lambda x: x.quantile(0.975)])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.rename(columns=lambda c: c.replace("<lambda_0>", "ci_low").replace("<lambda_1>", "ci_high")).reset_index()
summary.to_csv(SUMMARY_CSV, index=False)
print(summary.to_string(index=False))

             scope       kind                                 name  auroc_mean  auroc_ci_low  auroc_ci_high  auprc_mean  auprc_ci_low  auprc_ci_high
   task15_full_oof difference  stability_tmd_64_minus_stability_61    0.013332     -0.008062       0.034590    0.022852     -0.004981       0.050847
   task15_full_oof      model                         stability_61    0.628436      0.589204       0.668491    0.180265      0.139697       0.225586
   task15_full_oof      model                     stability_tmd_64    0.641768      0.601314       0.681513    0.203117      0.152077       0.258818
  task15_paired_am difference stability_tmd_64_minus_alphamissense   -0.004869     -0.048114       0.037900    0.038758     -0.006333       0.086163
  task15_paired_am      model                        alphamissense    0.649048      0.607946       0.690462    0.165904      0.129925       0.208147
  task15_paired_am      model                     stability_tmd_64    0.644180      0.604823       0.68256

## 17.2 Main-model decision rule

主模型判据预先固定为：以 Task 15 full-cohort OOF 的 XGBoost 64D 为 primary model。只有当 Task 16 中 TabPFN 或 ensemble 相对 XGBoost 的 AUROC 或 AUPRC paired difference CI 明确高于 0，且另一项指标没有明显恶化时，才升级主模型。单次点估计不触发模型替换。

In [3]:
decision_rows = summary[(summary["scope"] == "task16_paired_test") & (summary["kind"] == "difference")].copy()
print("Primary model decision")
print("  Retain XGBoost 64D as the primary model unless a challenger has a positive paired CI and no material trade-off.")
print(decision_rows.to_string(index=False))

Primary model decision
  Retain XGBoost 64D as the primary model unless a challenger has a positive paired CI and no material trade-off.
             scope       kind                           name  auroc_mean  auroc_ci_low  auroc_ci_high  auprc_mean  auprc_ci_low  auprc_ci_high
task16_paired_test difference   ensemble_64_minus_xgboost_64   -0.000741     -0.055511       0.052189    0.000763     -0.064852       0.064337
task16_paired_test difference     tabpfn_64_minus_xgboost_64   -0.021606     -0.111114       0.063034    0.039449     -0.081499       0.164685
task16_paired_test difference xgboost_64_minus_alphamissense    0.018203     -0.145470       0.182299    0.052306     -0.081576       0.205710


## 17.3 TMD robustness

比较 64D−61D 在各 fold、`in_TMD` subgroup 和 gene-level 的表现。Subgroup 样本若只有一个 class，则对应 AUROC 记为缺失。Gene-level 部分统计每个 gene 的平均 prediction change，用于识别增益是否由少数 genes 驱动；它不是 gene-level AUROC。

In [4]:
robust = task15.merge(tmd[["KEY", "in_TMD", "dist_to_nearest_TMD", "delta_membrane_insertion"]], on="KEY", how="left", validate="one_to_one")
assert robust["in_TMD"].notna().all()
robust["tmd_group"] = np.where(robust["in_TMD"] > 0, "in_TMD", "outside_TMD")

def compare_61_64(data, scope, subgroup):
    y_sub = data["Mislocalized"].to_numpy(dtype=int)
    auc61, ap61, _ = safe_metrics(y_sub, data["oof_stability_61"])
    auc64, ap64, _ = safe_metrics(y_sub, data["oof_stability_tmd_64"])
    return {"scope": scope, "subgroup": subgroup, "n": len(data), "positives": int(y_sub.sum()), "auroc_61": auc61, "auroc_64": auc64, "delta_auroc": auc64 - auc61, "auprc_61": ap61, "auprc_64": ap64, "delta_auprc": ap64 - ap61}

robustness_records = [compare_61_64(robust, "overall", "all")]
for fold, subset in robust.groupby("fold"):
    robustness_records.append(compare_61_64(subset, "fold", str(int(fold))))
for group, subset in robust.groupby("tmd_group"):
    robustness_records.append(compare_61_64(subset, "tmd_group", group))
robustness = pd.DataFrame(robustness_records)
robustness.to_csv(TMD_ROBUSTNESS_CSV, index=False)
print(robustness.to_string(index=False))

robust["prediction_delta_64_minus_61"] = robust["oof_stability_tmd_64"] - robust["oof_stability_61"]
gene_delta = robust.groupby("Gene").agg(n=("KEY", "size"), positives=("Mislocalized", "sum"), mean_delta=("prediction_delta_64_minus_61", "mean"), mean_in_TMD=("in_TMD", "mean")).sort_values("mean_delta", key=np.abs, ascending=False)
print("\nGenes with the largest absolute mean prediction change after adding TMD features:")
print(gene_delta.head(20).to_string())

    scope    subgroup    n  positives  auroc_61  auroc_64  delta_auroc  auprc_61  auprc_64  delta_auprc
  overall         all 2179        236  0.628641  0.642168     0.013527  0.175762  0.198098     0.022336
     fold           0  406         47  0.645884  0.660760     0.014876  0.186004  0.216510     0.030506
     fold           1  430         41  0.595899  0.631388     0.035488  0.147849  0.220034     0.072185
     fold           2  431         51  0.592415  0.648349     0.055934  0.197209  0.240184     0.042975
     fold           3  454         63  0.641619  0.634352    -0.007267  0.248633  0.229021    -0.019613
     fold           4  458         34  0.697558  0.674320    -0.023238  0.157227  0.169541     0.012314
tmd_group      in_TMD  151         39  0.703526  0.675824    -0.027701  0.443994  0.392566    -0.051427
tmd_group outside_TMD 2028        197  0.607363  0.615799     0.008436  0.147991  0.162250     0.014259

Genes with the largest absolute mean prediction change after ad

## 17.4 Exploratory error analysis

使用 Task 15 的 64D pooled OOF predictions。为生成 confusion categories，阈值在整套 OOF predictions 上选择为 F1 最大值，因此这部分只用于探索错误模式，不能作为独立 performance estimate。输出按 fold、source、TMD group、AlphaMissense availability 及可用 phenotype metadata 分层，并保存高置信 false positives/false negatives。

In [5]:
analysis = robust.copy()
optional_metadata = [c for c in ["label_5class", "Primary localization", "Secondary localization"] if c in features.columns and c not in analysis.columns]
if optional_metadata:
    analysis = analysis.merge(features[["KEY"] + optional_metadata], on="KEY", how="left", validate="one_to_one")

precision, recall, thresholds = precision_recall_curve(analysis["Mislocalized"], analysis["oof_stability_tmd_64"])
f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
threshold = float(thresholds[np.nanargmax(f1)])
analysis["predicted_positive"] = (analysis["oof_stability_tmd_64"] >= threshold).astype(int)
analysis["error_type"] = np.select(
    [
        (analysis["Mislocalized"] == 1) & (analysis["predicted_positive"] == 1),
        (analysis["Mislocalized"] == 0) & (analysis["predicted_positive"] == 0),
        (analysis["Mislocalized"] == 0) & (analysis["predicted_positive"] == 1),
        (analysis["Mislocalized"] == 1) & (analysis["predicted_positive"] == 0),
    ],
    ["TP", "TN", "FP", "FN"],
    default="unknown",
)
analysis["am_available"] = analysis["final_alphamissense_score"].notna()
print(f"Exploratory OOF F1-maximising threshold: {threshold:.6f}")
print(analysis["error_type"].value_counts().to_string())

summary_parts = []
stratifiers = ["fold", "source", "tmd_group", "am_available"] + optional_metadata
for stratifier in stratifiers:
    table = analysis.groupby(stratifier, dropna=False)["error_type"].value_counts().unstack(fill_value=0).reset_index()
    table.insert(0, "stratifier", stratifier)
    table = table.rename(columns={stratifier: "level"})
    for col in ["TP", "TN", "FP", "FN"]:
        if col not in table.columns:
            table[col] = 0
    table["sensitivity"] = table["TP"] / (table["TP"] + table["FN"]).replace(0, np.nan)
    table["specificity"] = table["TN"] / (table["TN"] + table["FP"]).replace(0, np.nan)
    table["precision"] = table["TP"] / (table["TP"] + table["FP"]).replace(0, np.nan)
    summary_parts.append(table)
error_summary = pd.concat(summary_parts, ignore_index=True)
error_summary.to_csv(ERROR_SUMMARY_CSV, index=False)

error_rows = analysis.loc[analysis["error_type"].isin(["FP", "FN"])].copy()
error_rows["error_confidence"] = np.where(error_rows["error_type"] == "FP", error_rows["oof_stability_tmd_64"], 1 - error_rows["oof_stability_tmd_64"])
error_rows = error_rows.sort_values("error_confidence", ascending=False)
error_rows.to_csv(ERROR_ROWS_CSV, index=False)
print("\nError summary:")
print(error_summary.to_string(index=False))
print(f"\nSaved: {BOOTSTRAP_CSV}")
print(f"Saved: {SUMMARY_CSV}")
print(f"Saved: {TMD_ROBUSTNESS_CSV}")
print(f"Saved: {ERROR_SUMMARY_CSV}")
print(f"Saved: {ERROR_ROWS_CSV}")

Exploratory OOF F1-maximising threshold: 0.277045
error_type
TN    1506
FP     437
FN     134
TP     102

Error summary:
  stratifier             level  FN  FP   TN  TP  sensitivity  specificity  precision
        fold                 0  24  72  287  23     0.489362     0.799443   0.242105
        fold                 1  24  76  313  17     0.414634     0.804627   0.182796
        fold                 2  28  93  287  23     0.450980     0.755263   0.198276
        fold                 3  42  83  308  21     0.333333     0.787724   0.201923
        fold                 4  16 113  311  18     0.529412     0.733491   0.137405
      source additional_benign   0   9   80   1     1.000000     0.898876   0.100000
      source              main 134 428 1426 101     0.429787     0.769148   0.190926
   tmd_group            in_TMD   9  51   61  30     0.769231     0.544643   0.370370
   tmd_group       outside_TMD 125 386 1445  72     0.365482     0.789186   0.157205
am_available             Fals